In [ ]:
import requests
import time
import json
import os
from datetime import datetime

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to assign category based on title
def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for word in keywords:
            if word in title:
                return category
    return None  # skip if no category matches


try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    if response.status_code == 200:
        story_ids = response.json()[:500]  # first 500 IDs
    else:
        print("Failed to fetch top stories")
        story_ids = []
except Exception as e:
    print("Error fetching top stories:", e)
    story_ids = []


collected_stories = []
category_count = {cat: 0 for cat in categories}

for category in categories:
    print(f"Fetching stories for category: {category}")
    
    for story_id in story_ids:
        # Stop if we already have 25 stories for this category
        if category_count[category] >= 25:
            break
        
        try:
            res = requests.get(ITEM_URL.format(story_id), headers=headers)
            
            if res.status_code != 200:
                print(f"Failed to fetch story {story_id}")
                continue
            
            story = res.json()
            
            
            if not story or "title" not in story:
                continue
            
            title = story.get("title", "")
            detected_category = get_category(title)
            
            
            if detected_category == category:
                
                collected_stories.append({
                    "post_id": story.get("id"),
                    "title": title.strip(),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                
                category_count[category] += 1
        
        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue
    
    
    time.sleep(2)



os.makedirs("data", exist_ok=True)

file_name = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

with open(file_name, "w", encoding="utf-8") as f:
    json.dump(collected_stories, f, indent=4)


print(f"\nCollected {len(collected_stories)} stories.")
print(f"Saved to {file_name}")